In [1]:
import itertools

# ─── Matrice des distances ───────────────────────────────────────────────────
dist = [
    [0, 10, 12, 11, 14, 12, 16, 15],
    [10,  0,  9, 15, 21,  8, 10, 12],
    [12,  9,  0,  7, 12, 14,  8, 11],
    [11, 15,  7,  0, 12, 10, 14,  9],
    [14, 21, 12, 12,  0,  8, 12, 16],
    [12,  8, 14, 10,  8,  0, 11, 10],
    [16, 10,  8, 14, 12, 11,  0,  9],
    [15, 12, 11,  9, 16, 10,  9,  0],
]
N = len(dist)

# ─── Coût d'un tour ──────────────────────────────────────────────────────────
def cout(tour):
    return sum(dist[tour[i]][tour[(i+1) % N]] for i in range(N))

# ─── Voisinage : toutes les transpositions (i,j) avec i < j ─────────────────
def voisinage(tour):
    """Retourne tous les voisins par échange de 2 villes (n(n-1)/2 = 28 mouvements)."""
    voisins = []
    for i, j in itertools.combinations(range(N), 2):
        nouveau = tour[:]
        nouveau[i], nouveau[j] = nouveau[j], nouveau[i]
        voisins.append(((i, j), nouveau))
    return voisins

# ─── Recherche Tabou ─────────────────────────────────────────────────────────
def recherche_tabou(tour_initial, T=3, nb_iter=3):
    """
    Paramètres :
      tour_initial : liste des villes (indices 0..7)
      T            : durée de la liste tabou (en itérations, critère d'aspiration = lever après T)
      nb_iter      : nombre d'itérations à effectuer
    """
    s_courant   = tour_initial[:]
    s_meilleur  = tour_initial[:]
    c_meilleur  = cout(s_meilleur)

    liste_tabou = []          # FIFO : stocke les mouvements (i,j) interdits
    historique  = []          # pour l'affichage

    print("=" * 75)
    print("RECHERCHE TABOU — TSP 8 villes")
    print("=" * 75)
    print(f"Solution initiale : {[v+1 for v in s_courant]} → Coût = {cout(s_courant)}")
    print(f"Durée liste tabou : T = {T}  |  Critère aspiration : lever si améliore s*")
    print(f"Voisinage         : transposition (swap) — {N*(N-1)//2} mouvements possibles")
    print("=" * 75)

    for iteration in range(1, nb_iter + 1):
        print(f"\n{'─'*75}")
        print(f"  ITÉRATION {iteration}")
        print(f"  Solution courante : {[v+1 for v in s_courant]}  |  Coût = {cout(s_courant)}")
        print(f"  Liste tabou       : {[(i+1, j+1) for (i,j) in liste_tabou]}")
        print(f"{'─'*75}")

        meilleur_voisin     = None
        meilleur_mouvement  = None
        meilleur_cout       = float('inf')
        tabou_leve          = False

        # Évaluation de tous les voisins
        for (i, j), voisin in voisinage(s_courant):
            c_voisin   = cout(voisin)
            est_tabou  = (i, j) in liste_tabou

            aspiration = est_tabou and c_voisin < c_meilleur   # critère d'aspiration

            statut = "TABOU" if est_tabou else "OK   "
            if aspiration:
                statut = "TABOU→LEVÉ (aspiration)"

            if not est_tabou or aspiration:
                if c_voisin < meilleur_cout:
                    meilleur_cout      = c_voisin
                    meilleur_voisin    = voisin
                    meilleur_mouvement = (i, j)
                    tabou_leve         = aspiration

        # Affichage du meilleur mouvement retenu
        i, j = meilleur_mouvement
        print(f"  ✔  Meilleur mouvement retenu : swap(V{i+1}, V{j+1})")
        print(f"     Nouveau tour : {[v+1 for v in meilleur_voisin]}")
        print(f"     Coût         : {meilleur_cout}")
        if tabou_leve:
            print(f"     ⚡ Critère d'aspiration appliqué — mouvement tabou levé !")

        # Mise à jour
        s_courant = meilleur_voisin[:]

        # Liste tabou FIFO : ajouter le mouvement, retirer si longueur > T
        liste_tabou.append(meilleur_mouvement)
        if len(liste_tabou) > T:
            retiré = liste_tabou.pop(0)
            print(f"     📤 Mouvement swap(V{retiré[0]+1}, V{retiré[1]+1}) retiré de la liste tabou (expiré)")

        # Mise à jour du meilleur global
        if meilleur_cout < c_meilleur:
            s_meilleur = s_courant[:]
            c_meilleur = meilleur_cout
            print(f"     🏆 Nouveau meilleur global trouvé ! Coût = {c_meilleur}")

        historique.append({
            'iter'      : iteration,
            'tour'      : [v+1 for v in s_courant],
            'cout'      : meilleur_cout,
            'mouvement' : (i+1, j+1),
            'tabou'     : [(a+1, b+1) for (a,b) in liste_tabou],
        })

    # ─── Résultats finaux ────────────────────────────────────────────────────
    print(f"\n{'=' * 75}")
    print("RÉSULTATS FINAUX")
    print(f"{'=' * 75}")
    print(f"{'Iter':<6} {'Tour':<42} {'Coût':<8} {'Mouvement':<14} {'Liste Tabou'}")
    print(f"{'─'*6} {'─'*42} {'─'*8} {'─'*14} {'─'*20}")
    print(f"{'0':<6} {str([v+1 for v in tour_initial]):<42} {cout(tour_initial):<8} {'—':<14} []")
    for h in historique:
        print(f"{h['iter']:<6} {str(h['tour']):<42} {h['cout']:<8} {str(h['mouvement']):<14} {h['tabou']}")

    print(f"\n{'─' * 75}")
    print(f"  Meilleure solution trouvée : {[v+1 for v in s_meilleur]}")
    print(f"  Coût optimal après {nb_iter} itérations : {c_meilleur}")
    print(f"{'=' * 75}")

    return s_meilleur, c_meilleur

# ─── Lancement ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Solution initiale naturelle : 1-2-3-4-5-6-7-8 (indices 0..7)
    s0 = [0, 1, 2, 3, 4, 5, 6, 7]

    # Paramètres de l'énoncé
    meilleure_solution, meilleur_cout = recherche_tabou(
        tour_initial = s0,
        T            = 3,   # taille liste tabou
        nb_iter      = 3,   # nombre d'itérations
    )

RECHERCHE TABOU — TSP 8 villes
Solution initiale : [1, 2, 3, 4, 5, 6, 7, 8] → Coût = 81
Durée liste tabou : T = 3  |  Critère aspiration : lever si améliore s*
Voisinage         : transposition (swap) — 28 mouvements possibles

───────────────────────────────────────────────────────────────────────────
  ITÉRATION 1
  Solution courante : [1, 2, 3, 4, 5, 6, 7, 8]  |  Coût = 81
  Liste tabou       : []
───────────────────────────────────────────────────────────────────────────
  ✔  Meilleur mouvement retenu : swap(V5, V6)
     Nouveau tour : [1, 2, 3, 4, 6, 5, 7, 8]
     Coût         : 80
     🏆 Nouveau meilleur global trouvé ! Coût = 80

───────────────────────────────────────────────────────────────────────────
  ITÉRATION 2
  Solution courante : [1, 2, 3, 4, 6, 5, 7, 8]  |  Coût = 80
  Liste tabou       : [(5, 6)]
───────────────────────────────────────────────────────────────────────────
  ✔  Meilleur mouvement retenu : swap(V2, V4)
     Nouveau tour : [1, 4, 3, 2, 6, 5, 7, 8]
     C